In [ ]:
import os
import pandas as pd

folder_path = r'C:\Users\user\CSE-CIC-IDS2018-Full'
file_path = os.path.join(folder_path, 'combined.csv')
# To prevent memory crashing
chunksize = 100_000

# Class distribution obtained from EDA 
class_percentages = {
    'Benign': 83.069712,
    'DDOS attack-HOIC': 4.22633,
    'DDoS attacks-LOIC-HTTP': 3.549504,
    'DoS attacks-Hulk': 2.845512,
    'Bot': 1.763020,
    'FTP-BruteForce': 1.191154,
    'SSH-Bruteforce': 1.155603,
    'Infilteration': 0.997560,
    'DoS attacks-SlowHTTPTest': 0.861763,
    'DoS attacks-GoldenEye': 0.255701,
    'DoS attacks-Slowloris': 0.067702,
    'DDOS attack-LOIC-UDP': 0.010657,
    'Brute Force -Web': 0.003764,
    'Brute Force -XSS': 0.001417,
    'SQL Injection': 0.000536
}

# Target sample size (10% of full dataset)
target_sample_size = 1623300
target_per_class = {label: int(p / 100 * target_sample_size) for label, p in class_percentages.items()}
collected = {label: [] for label in class_percentages}

# Read and sample by class
for chunk in pd.read_csv(file_path, chunksize=chunksize, low_memory=False):
    for label in class_percentages:
        rows = chunk[chunk['Label'] == label]
        needed = target_per_class[label] - sum(len(x) for x in collected[label])
        if needed > 0 and not rows.empty:
            sampled = rows.sample(n=min(len(rows), needed), random_state=42)
            collected[label].append(sampled)

# Combine and shuffle sample
sample_df = pd.concat([pd.concat(v) for v in collected.values()], ignore_index=True)
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

#  Save the sample 
output_sample_path = os.path.join(folder_path, "sample.csv")
sample_df.to_csv(output_sample_path, index=False)

print(f" Sampled dataset shape: {sample_df.shape}")

